# Tutorial 04: Discrete Time PINNs

In Tutorials 02–03 we used **continuous time** PINNs: the network takes $(x, t)$ as input and the PDE residual is enforced at random collocation points across the whole domain. This scales poorly — more collocation points mean more autograd calls per iteration.

**Discrete time PINNs** (Raissi et al. 2019, §3.2) take a different approach: encode the time-stepping scheme directly into the network structure. Given data at time $t^n$, the network predicts the solution at $t^{n+1}$ via an implicit Runge-Kutta (IRK) scheme — no collocation grid needed.

| | Continuous (Tutorial 03) | Discrete (Tutorial 04) |
|---|---|---|
| **Input** | $(x, t)$ | $x$ only |
| **Output** | $u(x,t)$ scalar | $q+1$ values (RK stages + final) |
| **Physics** | PDE residual at collocation | RK constraint equations |
| **Data loss** | MSE on IC/BC | None (all in `model_loss`) |

We demonstrate this on the **Allen-Cahn equation**, a nonlinear PDE that models phase separation.

---

### Table of Contents

- [Step 0: Setups](#step0)
- [Step 1: The Allen-Cahn equation](#step1)
- [Step 2: The discrete time PINN](#step2)
- [Step 3: Define the model](#step3)
- [Step 4: Train and evaluate](#step4)
- [Takeaway](#takeaway)
- [References](#references)

---

<a id='step0'></a>
### Step 0: Setups
---

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from utils import add_necessary_paths, check_torch
add_necessary_paths()

import utils.u04 as u
import talos as ta
import numpy as np
import torch

check_torch()
ta.set_seed()

<a id='step1'></a>
### Step 1: The Allen-Cahn equation
---

The Allen-Cahn equation describes phase separation with a cubic nonlinearity:

$$u_t = 0.0001\, u_{xx} + 5(u - u^3), \quad x \in [-1, 1], \quad t \in [0, 1]$$

with **periodic boundary conditions** ($u(-1,t) = u(1,t)$) and initial condition:

$$u(x, 0) = x^2 \cos(\pi x)$$

The small diffusion coefficient (0.0001) and the cubic reaction term create sharp interfaces — a challenging test for any numerical method.

We first generate a reference solution using scipy's `solve_ivp` with method of lines.

In [ ]:
# Generate reference solution
x_grid, t_grid, u_ref = u.solve_allen_cahn(n_x=512)
print(f'Spatial grid: {x_grid.shape}, Time grid: {t_grid.shape}, Solution: {u_ref.shape}')

# Visualize the full solution
u.plot_solution(x_grid, t_grid, u_ref)

Our task: given $N_n = 200$ noisy-free measurements at $t^n = 0.1$, predict the solution at $t^{n+1} = 0.9$ — a single large time step of $\Delta t = 0.8$.

In [ ]:
# (1) Extract reference snapshots at t=0.1 and t=0.9.
t_n, t_n1 = 0.1, 0.9
idx_tn = np.argmin(np.abs(t_grid - t_n))
idx_tn1 = np.argmin(np.abs(t_grid - t_n1))
u_at_tn = u_ref[:, idx_tn]
u_at_tn1 = u_ref[:, idx_tn1]

# (2) Subsample training points at t_n.
X_train, Y_train, train_idx = u.generate_training_data(x_grid, u_at_tn, n_train=200)
train_set = ta.Dataset(X_train, Y_train, name='u(x, t_n)')
train_set.report()

# (3) Visualize training data.
u.plot_training_data(x_grid, u_at_tn, X_train, Y_train, t_n)

<a id='step2'></a>
### Step 2: The discrete time PINN
---

#### Key idea

Instead of training a network on the full $(x, t)$ domain, we train one that takes **only $x$** and outputs $q + 1$ values:

$$\text{NN}(x) \;=\; \big[\hat{u}^{n+c_1}(x),\; \ldots,\; \hat{u}^{n+c_q}(x),\; \hat{u}^{n+1}(x)\big]$$

where $c_1, \ldots, c_q$ are the Gauss-Legendre nodes of a $q$-stage implicit Runge-Kutta method.

#### RK constraint

For a PDE $u_t = \mathcal{N}[u]$, the IRK scheme gives:

$$u^{n+c_j} = u^n + \Delta t \sum_{i=1}^{q} a_{ji}\, \mathcal{N}[u^{n+c_i}], \quad j = 1, \ldots, q$$
$$u^{n+1} = u^n + \Delta t \sum_{i=1}^{q} b_i\, \mathcal{N}[u^{n+c_i}]$$

Rearranging, we can **reconstruct** $u^n$ from each stage output:

$$u^n_j = \hat{u}^{n+c_j}(x) - \Delta t \sum_{i=1}^{q} a_{ji}\, \mathcal{N}[\hat{u}^{n+c_i}(x)]$$

#### Loss function

The loss has two terms — both live entirely in `model_loss`:

- **$\text{SSE}_n$**: Each reconstructed $u^n_j$ should match the training data $u^n(x)$
- **$\text{SSE}_b$**: Periodic boundary conditions — all outputs must satisfy $\hat{u}(-1) = \hat{u}(1)$

There is **no standard data loss** — the trainer runs with `loss_fn=None`.

<a id='step3'></a>
### Step 3: Define the model
---

We subclass `MLP` with $q+1$ outputs and override `model_loss` to enforce the IRK constraints. The nonlinear operator for the Allen-Cahn equation is:

$$\mathcal{N}[u] = 0.0001\, u_{xx} + 5(u - u^3)$$

Computing $u_{xx}$ for each stage output requires `torch.autograd.grad` — we do a fresh forward pass with `requires_grad_(True)` on the input.

In [ ]:
class AllenCahnPINN(ta.model.torch_zoo.MLP):
  """Discrete time PINN for the Allen-Cahn equation."""

  def __init__(self, hidden, q=20, dt=0.8):
    super().__init__(1, hidden, q + 1, activation='tanh')
    self.q = q
    self.dt = dt
    # (1) Register IRK weights as buffers (not trainable).
    a, b, c = u.irk_weights(q)
    self.register_buffer('irk_a', torch.tensor(a, dtype=torch.float32))
    self.register_buffer('irk_b', torch.tensor(b, dtype=torch.float32))

  def _nonlinear_op(self, x, u_vals):
    """Compute N[u] = 0.0001 * u_xx + 5*(u - u^3) for each stage.

    Args:
      x: (N, 1) input with requires_grad=True.
      u_vals: (N, q+1) network outputs.

    Returns:
      N_vals: (N, q+1) nonlinear operator applied to each output column.
    """
    N_vals = []
    for j in range(u_vals.shape[1]):
      u_j = u_vals[:, j:j+1]
      # (1) First derivative.
      u_x = torch.autograd.grad(u_j, x, torch.ones_like(u_j),
                                create_graph=True)[0]
      # (2) Second derivative.
      u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x),
                                 create_graph=True)[0]
      # (3) N[u] = 0.0001 * u_xx + 5*(u - u^3).
      N_j = 0.0001 * u_xx + 5.0 * (u_j - u_j**3)
      N_vals.append(N_j)
    return torch.cat(N_vals, dim=1)  # (N, q+1)

  def model_loss(self, X, outputs, Y):
    """Discrete time PINN loss: SSE_n (RK constraint) + SSE_b (periodic BC).

    Args:
      X: (N, 1) training x-points (as tensors from trainer).
      outputs: (N, q+1) network outputs (ignored — we recompute with grad).
      Y: (N, 1) known u(x, t_n) values.
    """
    # (1) Fresh forward pass with grad-enabled input.
    x = X.detach().clone().requires_grad_(True)
    u_pred = self.forward(x)  # (N, q+1)

    # (2) Compute nonlinear operator for all stages.
    N_all = self._nonlinear_op(x, u_pred)  # (N, q+1)
    N_stages = N_all[:, :self.q]  # (N, q) — only the q RK stages

    # (3) Reconstruct u^n from each RK stage.
    # u^n_j = u_pred[:, j] - dt * sum_i a[j,i] * N[u^{n+c_i}]
    u_stages = u_pred[:, :self.q]  # (N, q)
    u_n_reconstructed = u_stages - self.dt * (N_stages @ self.irk_a.T)  # (N, q)

    # (3.1) Reconstruct u^n from the final output using b weights.
    u_final = u_pred[:, self.q:self.q+1]  # (N, 1)
    u_n_from_final = u_final - self.dt * (N_stages @ self.irk_b.unsqueeze(1))  # (N, 1)
    u_n_reconstructed = torch.cat([u_n_reconstructed, u_n_from_final], dim=1)  # (N, q+1)

    # (4) SSE_n: all reconstructed u^n should match training data.
    sse_n = torch.mean((u_n_reconstructed - Y) ** 2)

    # (5) SSE_b: periodic BC — u(-1) = u(1) for all outputs.
    x_left = torch.tensor([[-1.0]], device=X.device)
    x_right = torch.tensor([[1.0]], device=X.device)
    u_left = self.forward(x_left)   # (1, q+1)
    u_right = self.forward(x_right)  # (1, q+1)
    sse_b = torch.mean((u_left - u_right) ** 2)

    return sse_n + sse_b

In [ ]:
# Initialize model: 4 hidden layers of 128 neurons, q=20 RK stages
q = 20
dt = 0.8  # t_n=0.1 -> t_{n+1}=0.9
pinn = AllenCahnPINN([128, 128, 128, 128], q=q, dt=dt)
print(pinn)

<a id='step4'></a>
### Step 4: Train and evaluate
---

We train with `loss_fn=None` — there is no standard data loss. The entire training signal comes from `model_loss`, which enforces the IRK constraints and periodic BCs.

In [ ]:
# Train with no data loss (model_loss only)
trainer = u.get_trainer(pinn, optimizer='adam', lr=1e-3, print_every=1000)
trainer.train(train_set, max_iterations=10000)

In [ ]:
# Predict on the full spatial grid
X_dense = x_grid.reshape(-1, 1)
Y_pred_all = pinn.predict(ta.Dataset(X_dense))  # (n_x, q+1)

# Last column is u^{n+1}(x) = u(x, 0.9)
u_pred_tn1 = Y_pred_all[:, -1]
print(f'Prediction shape: {Y_pred_all.shape}')
print(f'u^{{n+1}} shape: {u_pred_tn1.shape}')

In [ ]:
# Compare prediction at t=0.9 with reference
u.plot_prediction(x_grid, u_at_tn1, u_pred_tn1, t_pred=t_n1)

# Compute relative L2 error
rel_l2 = np.linalg.norm(u_pred_tn1 - u_at_tn1) / np.linalg.norm(u_at_tn1)
print(f'Relative L2 error at t={t_n1}: {rel_l2:.4e}')

<a id='takeaway'></a>
### Takeaway

Discrete time PINNs encode time-stepping directly into the network architecture rather than enforcing the PDE at collocation points. Key differences from the continuous approach:

- **No collocation grid** — physics is enforced through Runge-Kutta structure
- **`loss_fn=None`** — all training signal comes from `model_loss`
- **Large time steps** — high-order IRK schemes (here $q = 20$) allow stepping from $t = 0.1$ to $t = 0.9$ in one shot

**When to use which:**
- *Continuous PINNs*: general-purpose, flexible boundary conditions, but collocation cost grows with domain size
- *Discrete PINNs*: natural for time-stepping problems, efficient for large $\Delta t$, but require known data at a time snapshot

<a id='references'></a>
### References

**Raissi, M., Perdikaris, P., & Karniadakis, G. E.** (2019). Physics-informed neural networks: A deep learning framework for solving forward and inverse problems involving nonlinear partial differential equations. *Journal of Computational Physics*, 378, 686–707. [doi:10.1016/j.jcp.2018.10.045](https://doi.org/10.1016/j.jcp.2018.10.045)